# Solution: Simple PID Controller


In [1]:
# Import standard libraries
import math
from pathlib import Path
import subprocess
import sys
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

ModuleNotFoundError: No module named 'mujoco'

In [ ]:
# Settings
MJCF_PATH = Path.cwd().parents[1] / "mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml"
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 250          # Refresh the status display twice per second

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
LEFT_WHEEL_POS = "left_wheel_pos"
RIGHT_WHEEL_POS = "right_wheel_pos"
LEFT_WHEEL_VEL = "left_wheel_vel"
RIGHT_WHEEL_VEL = "right_wheel_vel"

In [ ]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [ ]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [ ]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

In [ ]:
# Filter and PID coefficients (tune these)
ALPHA = 0.99
KP = 7.0      # Proportional constant
KD = 0.5      # Derivative constant

# When the robot has tipped over into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Run the native viewer under mjpython on macOS; passive viewing is unsupported in a regular Jupyter kernel.
if sys.platform == "darwin":
    mjpython = Path(sys.executable).with_name("mjpython")
    if not mjpython.is_file():
        raise FileNotFoundError(f"mjpython not found next to the active kernel: {mjpython}. Select the project .venv kernel.")
    script_path = Path.cwd() / "solution_simple_pid.py"
    print("Launching the viewer with mjpython. This cell runs until the viewer is closed.")
    subprocess.run([str(mjpython), str(script_path)], check=True)
else:
    # Resets simulation data to defaults
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

    # Launch MuJoCo simulator and GUI
    steps = 0
    pitch = 0.0
    tipped = False
    prev_time = 0.0
    with mujoco.viewer.launch_passive(model, data) as viewer:
        # Define free-look camera (control with mouse), looking at robot's back-right
        viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        viewer.cam.lookat[:] = [0, 0, 0.05]
        viewer.cam.distance  = 0.8
        viewer.cam.azimuth   = 45
        viewer.cam.elevation = -25

        # Simulation loop
        while viewer.is_running():
            step_start = time.time()

            # Detect MuJoCo reset and reset variables
            if data.time < prev_time:
                tipped = False
                mujoco.mj_forward(model, data)
                accel_x, _, accel_z = data.sensor(IMU_ACCEL).data
                pitch = -1 * math.atan2(accel_x, accel_z)
            prev_time = data.time

            # Read sensors
            accel_x, _, accel_z = data.sensor(IMU_ACCEL).data
            pitch_rate = data.sensor(IMU_GYRO).data[1]

            # Estimate pitch from accelerometer and gyroscope
            accel_pitch = -1 * math.atan2(accel_x, accel_z)
            pitch = ALPHA * (pitch + (pitch_rate * model.opt.timestep)) + (1 - ALPHA) * accel_pitch

            # Stop the wheels if the robot tips over
            if abs(pitch) > TIP_THRESHOLD:
                tipped = True

            if tipped:
                motor_spd_target = 0.0
            else:
                # PD control (no I term needed)
                motor_spd_target = clamp((KP * pitch) + (KD * pitch_rate))

            # Update motor speeds
            data.ctrl[left_motor_id] = motor_spd_target
            data.ctrl[right_motor_id] = motor_spd_target

            # Advance simulation by one step
            mujoco.mj_step(model, data)
            viewer.sync()

            # Refresh the compact status display twice per second
            steps += 1
            if steps % PRINT_EVERY == 0:
                clear_output(wait=True)
                print("MuJoCo: PID balance controller")
                print("Readout refresh: 2 Hz | angles and rates rounded to 3 decimals")
                print(f"Accel pitch:    {accel_pitch:+.3f} rad")
                print(f"Filtered pitch: {pitch:+.3f} rad")
                print(f"Pitch rate:     {pitch_rate:+.3f} rad/s")
                print(f"Motor command:  {motor_spd_target:+.3f}")
                print(f"Tipped:         {tipped}")

            # Keep simulation time close to real time
            slack = model.opt.timestep - (time.time() - step_start)
            if slack > 0:
                time.sleep(slack)